In [ ]:
"""
Physics-Informed Neural Network (PINN) for Gravitational Lensing Classification
Uses the gravitational lensing equation as a physics prior within the network
architecture to classify strong lensing images into three classes:
  0: No substructure
  1: Subhalo (sphere) substructure
  2: Vortex substructure

Physics: The gravitational lensing equation relates the source position (beta)
to the image position (theta) via the deflection angle (alpha):
    beta = theta - alpha(theta)
where alpha depends on the mass distribution of the lens.

For a singular isothermal sphere (SIS):
    alpha(theta) = theta_E * theta / |theta|
For substructure perturbations, the deflection gets additional terms.

This PINN incorporates:
1. A physics-based feature extraction layer that computes lensing-relevant
   quantities (convergence, shear, deflection field estimates) from the image.
2. A residual connection that enforces the lens equation as a soft constraint.
3. A physics-informed loss that penalizes violations of lensing symmetry.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

IMG_SIZE = 64  # Resize from 150 to 64 for faster training
BATCH_SIZE = 64
NUM_CLASSES = 3
EPOCHS = 40
LR = 1e-3
PHYSICS_LOSS_WEIGHT = 0.1
NUM_WORKERS = 0  # Safe for macOS

In [ ]:
# Dataset
class LensingDataset(Dataset):
    """Loads .npy gravitational lensing images with on-the-fly augmentation."""

    CLASS_MAP = {'no': 0, 'sphere': 1, 'vort': 2}

    def __init__(self, root_dir, augment=False):
        self.samples = []
        self.augment = augment
        for class_name, label in self.CLASS_MAP.items():
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if fname.endswith('.npy'):
                    self.samples.append((os.path.join(class_dir, fname), label))
        print(f"  Loaded {len(self.samples)} samples from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.load(path).astype(np.float32)  # (1, 150, 150)

        if img.ndim == 2:
            img = img[np.newaxis]

        # Resize to IMG_SIZE using simple interpolation
        img_tensor = torch.from_numpy(img).unsqueeze(0)  # (1, 1, 150, 150)
        img_tensor = F.interpolate(img_tensor, size=(IMG_SIZE, IMG_SIZE),
                                   mode='bilinear', align_corners=False)
        img_tensor = img_tensor.squeeze(0)  # (1, IMG_SIZE, IMG_SIZE)

        # Data augmentation
        if self.augment:
            # Random horizontal flip
            if torch.rand(1).item() > 0.5:
                img_tensor = torch.flip(img_tensor, [2])
            # Random vertical flip
            if torch.rand(1).item() > 0.5:
                img_tensor = torch.flip(img_tensor, [1])
            # Random 90-degree rotation
            k = torch.randint(0, 4, (1,)).item()
            if k > 0:
                img_tensor = torch.rot90(img_tensor, k, [1, 2])
            # Small noise injection
            img_tensor = img_tensor + 0.01 * torch.randn_like(img_tensor)
            img_tensor = torch.clamp(img_tensor, 0, 1)

        return img_tensor, label

In [ ]:
# Physics Layers — Gravitational Lensing
class GravitationalLensingLayer(nn.Module):
    """
    Computes physics-informed features from gravitational lensing images.

    The gravitational lensing equation: beta = theta - alpha(theta)
    where alpha = grad(psi) and psi is the lensing potential.

    The convergence kappa = 0.5 * laplacian(psi) = Sigma / Sigma_cr
    The shear components: gamma1 = 0.5*(psi_11 - psi_22), gamma2 = psi_12

    From the image (surface brightness), we can estimate:
    - Convergence-like features (related to mass distribution)
    - Shear-like features (related to tidal gravitational field)
    - Deflection field estimates
    - Magnification estimates: mu = 1/((1-kappa)^2 - |gamma|^2)
    """

    def __init__(self, img_size):
        super().__init__()
        self.img_size = img_size

        # Create coordinate grids (theta_1, theta_2) for the image plane
        coords = torch.linspace(-1, 1, img_size)
        theta_1, theta_2 = torch.meshgrid(coords, coords, indexing='ij')
        self.register_buffer('theta_1', theta_1.unsqueeze(0).unsqueeze(0))
        self.register_buffer('theta_2', theta_2.unsqueeze(0).unsqueeze(0))

        # Radial distance from center (for SIS-like computations)
        r = torch.sqrt(theta_1**2 + theta_2**2 + 1e-8)
        self.register_buffer('r_grid', r.unsqueeze(0).unsqueeze(0))

        # Sobel filters for gradient computation (estimate deflection field)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                               dtype=torch.float32).reshape(1, 1, 3, 3) / 8.0
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                               dtype=torch.float32).reshape(1, 1, 3, 3) / 8.0
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

        # Laplacian filter (for convergence estimation)
        laplacian = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]],
                                 dtype=torch.float32).reshape(1, 1, 3, 3)
        self.register_buffer('laplacian_kernel', laplacian)

        # Learnable Einstein radius (key physical parameter)
        self.theta_E = nn.Parameter(torch.tensor(0.5))

        # Learnable parameters for the lens model
        self.lens_strength = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        """
        x: (B, 1, H, W) - input lensing image (surface brightness)
        Returns: (B, 8, H, W) - physics-informed feature maps
        """
        B = x.shape[0]

        # 1. Estimate gradients (proxy for deflection field components)
        #    In lensing, the deflection alpha = grad(psi)
        #    The image brightness gradient relates to the source structure
        alpha_1 = F.conv2d(x, self.sobel_x, padding=1)  # d/dx
        alpha_2 = F.conv2d(x, self.sobel_y, padding=1)  # d/dy

        # 2. Estimate convergence kappa ~ 0.5 * laplacian(psi)
        #    We use the image Laplacian as a proxy for convergence
        kappa = F.conv2d(x, self.laplacian_kernel, padding=1)

        # 3. Estimate shear components
        #    gamma1 ~ 0.5 * (d^2/dx^2 - d^2/dy^2) psi
        #    gamma2 ~ d^2/(dxdy) psi
        #    Approximate using second derivatives of the image
        gamma_1 = F.conv2d(alpha_1, self.sobel_x, padding=1)  # d^2/dx^2
        gamma_2_part = F.conv2d(alpha_1, self.sobel_y, padding=1)  # d^2/dxdy
        gamma_1 = gamma_1 - F.conv2d(alpha_2, self.sobel_y, padding=1)  # - d^2/dy^2
        gamma_1 = 0.5 * gamma_1
        gamma_2 = gamma_2_part

        # 4. Magnification: mu = 1/((1-kappa)^2 - gamma^2)
        gamma_sq = gamma_1**2 + gamma_2**2
        denom = (1 - kappa)**2 - gamma_sq
        mu = 1.0 / (denom + 1e-6 * torch.sign(denom + 1e-10))
        mu = torch.clamp(mu, -50, 50)  # Clip extreme magnifications

        # 5. Apply lens equation: beta = theta - theta_E * alpha_hat
        #    Compute the source-plane mapping
        theta_E = torch.abs(self.theta_E)
        r = self.r_grid.expand(B, -1, -1, -1)

        # SIS deflection: alpha = theta_E * theta/|theta|
        sis_alpha_1 = theta_E * self.theta_1 / r
        sis_alpha_2 = theta_E * self.theta_2 / r

        # Residual between image gradient and SIS model
        # This captures substructure deviations from smooth SIS
        residual_1 = alpha_1 * self.lens_strength - sis_alpha_1.expand(B, -1, -1, -1)
        residual_2 = alpha_2 * self.lens_strength - sis_alpha_2.expand(B, -1, -1, -1)
        residual_mag = torch.sqrt(residual_1**2 + residual_2**2 + 1e-8)

        # Stack all physics features
        features = torch.cat([
            x,             # Original image
            alpha_1,       # Deflection x-component
            alpha_2,       # Deflection y-component
            kappa,         # Convergence
            gamma_1,       # Shear component 1
            gamma_2,       # Shear component 2
            mu,            # Magnification
            residual_mag,  # Substructure residual (key discriminant!)
        ], dim=1)  # (B, 8, H, W)

        return features

In [ ]:
class LensEquationConstraint(nn.Module):
    """
    Computes a soft physics loss enforcing the lens equation.

    For images with no substructure, the residual from SIS should be small.
    For images with substructure, the residual pattern should be structured.

    This layer computes features that quantify how well the image matches
    a smooth lens model, with deviations indicating substructure.
    """

    def __init__(self, img_size):
        super().__init__()
        coords = torch.linspace(-1, 1, img_size)
        theta_1, theta_2 = torch.meshgrid(coords, coords, indexing='ij')
        r = torch.sqrt(theta_1**2 + theta_2**2 + 1e-8)
        self.register_buffer('r_grid', r)
        self.register_buffer('theta_1', theta_1)
        self.register_buffer('theta_2', theta_2)

        # Azimuthal angle for detecting angular substructure (vortex)
        phi = torch.atan2(theta_2, theta_1)
        self.register_buffer('phi_grid', phi)

    def compute_physics_loss(self, images, predictions):
        """
        Compute physics-informed regularization loss.

        For predicted class 0 (no substructure): image should be azimuthally
        symmetric → penalize angular variations.

        The loss encourages the network to learn physically consistent features.
        """
        B = images.shape[0]

        # Compute radial profile variance (should be low for smooth lenses)
        # Use the image intensity in radial bins
        r_flat = self.r_grid.reshape(-1)
        n_bins = 8
        bin_edges = torch.linspace(0, 1.42, n_bins + 1, device=images.device)

        radial_variance = torch.zeros(B, device=images.device)
        for i in range(n_bins):
            mask = ((r_flat >= bin_edges[i]) & (r_flat < bin_edges[i+1]))
            if mask.sum() > 1:
                bin_pixels = images[:, 0].reshape(B, -1)[:, mask]
                radial_variance += bin_pixels.var(dim=1)

        # Predictions softmax
        probs = F.softmax(predictions, dim=1)

        # For class 0 (no substructure), radial variance should be low
        # Weight the variance penalty by the predicted probability of class 0
        physics_loss = (probs[:, 0] * radial_variance).mean()

        return physics_loss

In [ ]:
# Network Architecture
class ResidualBlock(nn.Module):
    """Standard residual block with batch normalization."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.gelu(out)

In [ ]:
class PINNLensingClassifier(nn.Module):
    """
    Physics-Informed Neural Network for gravitational lensing classification.

    Architecture:
    1. GravitationalLensingLayer: Extracts physics features (convergence,
       shear, deflection, magnification, substructure residuals)
    2. CNN backbone: Processes both raw and physics features
    3. LensEquationConstraint: Provides physics-based regularization loss
    4. Classifier head with dropout for robustness
    """

    def __init__(self, img_size=IMG_SIZE, num_classes=NUM_CLASSES):
        super().__init__()

        # Physics layers
        self.physics_layer = GravitationalLensingLayer(img_size)
        self.lens_constraint = LensEquationConstraint(img_size)

        # Physics feature encoder (8 channels from physics layer)
        self.physics_encoder = nn.Sequential(
            nn.Conv2d(8, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )

        # Raw image encoder
        self.raw_encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )

        # Fusion and deeper feature extraction (64 channels = 32 physics + 32 raw)
        self.backbone = nn.Sequential(
            ResidualBlock(64, 64),
            nn.MaxPool2d(2),              # 64 -> 32
            ResidualBlock(64, 128),
            nn.MaxPool2d(2),              # 32 -> 16
            ResidualBlock(128, 256),
            nn.MaxPool2d(2),              # 16 -> 8
            ResidualBlock(256, 256),
            ResidualBlock(256, 256),
            nn.AdaptiveAvgPool2d(1),      # Global average pool
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        # Extract physics-informed features
        physics_features = self.physics_layer(x)     # (B, 8, H, W)
        physics_encoded = self.physics_encoder(physics_features)  # (B, 32, H, W)

        # Encode raw image
        raw_encoded = self.raw_encoder(x)            # (B, 32, H, W)

        # Fuse physics and raw features
        fused = torch.cat([physics_encoded, raw_encoded], dim=1)  # (B, 64, H, W)

        # Deep feature extraction
        features = self.backbone(fused)              # (B, 256, 1, 1)

        # Classification
        logits = self.classifier(features)           # (B, num_classes)

        return logits

    def compute_physics_loss(self, images, predictions):
        return self.lens_constraint.compute_physics_loss(images, predictions)

In [ ]:
# Training Loop
def train_one_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()

        # Forward pass
        logits = model(images)

        # Classification loss
        ce_loss = F.cross_entropy(logits, labels, label_smoothing=0.05)

        # Physics-informed loss
        physics_loss = model.compute_physics_loss(images, logits)

        # Combined loss
        loss = ce_loss + PHYSICS_LOSS_WEIGHT * physics_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


@torch.no_grad()

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        total_loss += loss.item() * images.size(0)
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        probs = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    return avg_loss, accuracy, all_probs, all_labels


@torch.no_grad()

In [ ]:
def tta_evaluate(model, loader):
    """16-fold Test-Time Augmentation: average predictions over all flips × rotations."""
    model.eval()
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        probs = torch.zeros(images.size(0), NUM_CLASSES, device=DEVICE)
        n = 0
        for flip_h in [False, True]:
            for flip_v in [False, True]:
                for rot_k in range(4):
                    aug = images.clone()
                    if flip_h:
                        aug = torch.flip(aug, [3])
                    if flip_v:
                        aug = torch.flip(aug, [2])
                    if rot_k:
                        aug = torch.rot90(aug, rot_k, [2, 3])
                    logits = model(aug)
                    probs += F.softmax(logits, dim=1)
                    n += 1
        probs /= n
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return np.concatenate(all_probs), np.concatenate(all_labels)

In [ ]:
# ROC Curve and AUC Computation
def plot_roc_curves(labels, probs, save_path='roc_curves.png'):
    """Plot ROC curves for each class and compute AUC scores."""
    class_names = ['No Substructure', 'Subhalo (Sphere)', 'Vortex']
    n_classes = len(class_names)

    # Binarize labels
    labels_bin = label_binarize(labels, classes=[0, 1, 2])

    plt.figure(figsize=(10, 8))
    colors = ['#2196F3', '#FF5722', '#4CAF50']
    aucs = {}

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr)
        aucs[class_names[i]] = roc_auc
        plt.plot(fpr, tpr, color=colors[i], lw=2.5,
                 label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

    # Compute macro-average ROC
    all_fpr = np.unique(np.concatenate([
        roc_curve(labels_bin[:, i], probs[:, i])[0] for i in range(n_classes)
    ]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probs[:, i])
        mean_tpr += np.interp(all_fpr, fpr, tpr)
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, color='navy', lw=3, linestyle='--',
             label=f'Macro-Average (AUC = {macro_auc:.4f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')
    plt.xlim([-0.01, 1.01])
    plt.ylim([-0.01, 1.01])
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title('PINN Gravitational Lensing Classifier — ROC Curves', fontsize=16)
    plt.legend(loc='lower right', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  ROC curve saved to {save_path}")

    # Also compute OvR AUC
    macro_auc_ovr = roc_auc_score(labels_bin, probs, average='macro', multi_class='ovr')

    return aucs, macro_auc_ovr

In [ ]:
def plot_training_history(train_losses, val_losses, train_accs, val_accs,
                          save_path='training_history.png'):
    """Plot training and validation loss/accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(train_losses) + 1)

    ax1.plot(epochs, train_losses, 'b-', lw=2, label='Train Loss')
    ax1.plot(epochs, val_losses, 'r-', lw=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training & Validation Loss', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_accs, 'b-', lw=2, label='Train Accuracy')
    ax2.plot(epochs, val_accs, 'r-', lw=2, label='Val Accuracy')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training & Validation Accuracy', fontsize=14)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Training history saved to {save_path}")

In [ ]:
# Main
def main():
    print("=" * 70)
    print("PINN Gravitational Lensing Classifier")
    print("=" * 70)

    # Data loading
    base_dir = os.path.dirname(os.path.abspath(__file__))
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'val')

    print("\nLoading datasets...")
    train_dataset = LensingDataset(train_dir, augment=True)
    val_dataset = LensingDataset(val_dir, augment=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=NUM_WORKERS,
                              pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS,
                            pin_memory=True)

    # Model
    model = PINNLensingClassifier(img_size=IMG_SIZE, num_classes=NUM_CLASSES).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    print(f"  Learnable Einstein radius (theta_E): {model.physics_layer.theta_E.item():.4f}")

    # Optimizer & Scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS,
                                                            eta_min=1e-6)

    # Training
    print(f"\nTraining for {EPOCHS} epochs...")
   

    best_val_acc = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(1, EPOCHS + 1):
        start = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer,
                                                 scheduler, epoch)
        val_loss, val_acc, val_probs, val_labels = evaluate(model, val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        elapsed = time.time() - start

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(base_dir, 'best_pinn_model.pt'))
            marker = ' *'
        else:
            marker = ''

        print(f"Epoch {epoch:2d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
              f"LR: {scheduler.get_last_lr()[0]:.2e} | "
              f"{elapsed:.1f}s{marker}")

    print("-" * 70)
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    print(f"Learned Einstein radius: {model.physics_layer.theta_E.item():.4f}")
    print(f"Learned lens strength: {model.physics_layer.lens_strength.item():.4f}")

    # Load best model for final evaluation
    model.load_state_dict(torch.load(os.path.join(base_dir, 'best_pinn_model.pt'),
                                     weights_only=True))
    val_loss, val_acc, val_probs, val_labels = evaluate(model, val_loader)

    # ── 16-fold Test-Time Augmentation ────────────────────────────────
    print("\nRunning 16-fold TTA (flips × rotations)...")
    tta_probs, tta_labels = tta_evaluate(model, val_loader)
    tta_preds = tta_probs.argmax(axis=1)
    tta_acc = 100.0 * (tta_preds == tta_labels).sum() / len(tta_labels)

    # Classification report
    
   
    print("Final Evaluation on Validation Set")
    print(f"  Accuracy (no TTA):  {val_acc:.2f}%")
    print(f"  Accuracy (16× TTA): {tta_acc:.2f}%\n")
    class_names = ['No Substructure', 'Subhalo (Sphere)', 'Vortex']
    print(classification_report(tta_labels, tta_preds, target_names=class_names, digits=4))

    # ROC curves and AUC (using TTA probabilities for best scores)
    aucs, macro_auc = plot_roc_curves(tta_labels, tta_probs,
                                       save_path=os.path.join(base_dir, 'roc_curves.png'))
    print(f"\nAUC Scores (with TTA):")
    for cls_name, auc_val in aucs.items():
        print(f"  {cls_name}: {auc_val:.4f}")
    print(f"  Macro-Average AUC: {macro_auc:.4f}")

    # Training history plot
    plot_training_history(train_losses, val_losses, train_accs, val_accs,
                          save_path=os.path.join(base_dir, 'training_history.png'))

    print(f"\nAll outputs saved to {base_dir}")

In [ ]:
if __name__ == '__main__':
    main()